# Species Richness by Location Map

This notebook generates and exports an HTML format interactive map of species richness at reported locations. To use it, update the country and, optionally, the year in the first code cell, below, before running the notebook.

If the year is specified, only locations reported in that year will be included.

If the year is left blank, all locations reported in the specified country for all time will be included.

In [ ]:
# Year to report on. Optional but if specified, should be in YYYY format e.g. 2025
year = "2026"

# Country to report on
country = "United Kingdom"

# Initial Zoom Level for the map
zoom = 6

In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb

In [ ]:
import sqlparse

# Select the query file based on whether the year is specified
query_file_name = "richness_by_year.sql" if year else "richness.sql"

# Construct the query
query = construct_query(query_file_name, {
    "YEAR": year,
    "COUNTRY": country
})

# Show a pretty-printed form of the query
print(sqlparse.format(query, reindent=True, keyword_case='upper'))

In [ ]:
# Connect to the database and execute the query and read the results into a dataframe
df = query_data(query)

# Check there is some data
if not df.shape[0]:
    message = f"No data found for country '{country}'"
    if year:
        message += f" during '{year}'"
    raise ValueError(message)

In [ ]:
import pandas as pd

# Make sure the latitude and longitude have been read as numbers, not strings
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

# Calculate the data for the richness chart
richness = (
    df.groupby(["Location", "Latitude", "Longitude"])["Species"]
    .nunique()
    .reset_index()
)

richness.columns = ["Location", "Latitude", "Longitude", "Richness"]

# Print the richness data
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    display(richness)

In [ ]:
import re

# Create the folder to hold exported reports
export_folder_path = get_export_folder_path()

# Export the richness data to Excel
clean_country = clean_string(country)
export_file_name = f"{year}-{clean_country}-Richness" if year else f"{clean_country}-Richness"
export_to_spreadsheet(export_folder_path, f"{export_file_name}.xlsx", {
    "Location Richness": richness
})

In [ ]:
import folium
from folium import CircleMarker
from folium.plugins import MarkerCluster

import os
from urllib.parse import quote

api_key = os.environ.get("CARTODB_API_KEY")
if not api_key:
    raise RuntimeError("Set CARTODB_API_KEY before starting Jupyter.")

tile_url = (
    "https://basemaps.cartocdn.com/rastertiles/light_all/{z}/{x}/{y}.png"
    f"?key={quote(api_key, safe='')}"
)

richness_map = folium.Map(
    location=[richness["Latitude"].mean(), richness["Longitude"].mean()],
    zoom_start=zoom,
    tiles=tile_url,
    attr=(
        '&copy; <a href="https://www.openstreetmap.org/copyright">'
        'OpenStreetMap</a> contributors '
        '&copy; <a href="https://carto.com/attributions">CARTO</a>'
    ),
)

# Optional: cluster markers if you have many locations
marker_cluster = MarkerCluster().add_to(richness_map)

# Normalize marker size based on richness
max_richness = richness["Richness"].max()

for _, row in richness.iterrows():
    CircleMarker(
        location=(row["Latitude"], row["Longitude"]),
        radius=5 + 10 * (row["Richness"] / max_richness),  # Size scaled by richness
        color="blue",
        fill=True,
        fill_color="green",
        fill_opacity=0.6,
        popup=f"<b>{row['Location']}</b><br>Richness: {row['Richness']}",
    ).add_to(marker_cluster)


# Export the map to HTML
export_file_path = export_folder_path / f"{export_file_name}.html"
richness_map.save(export_file_path.absolute())

# Show the map
richness_map